<a href="https://colab.research.google.com/github/pilargarlop/GithubProject/blob/main/Teachable_Machine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
1. El HTML index.html
Es el que coloca todas las piezas en la pantalla para que el usuario pueda interactuar:

Los scripts de arriba: Carga las librerías de TensorFlow y Teachable Machine desde internet. Sin esto, el navegador no sabe procesar IA.

El hueco de la cámara #webcam-container: Es un cuadro vacío donde luego el JavaScript mete el vídeo.

Caja de resultados: Donde pone la etiqueta de lo que ha visto y el porcentaje de acierto la confianza.

Botones: Para arrancar la cámara, pararla y ajustar el porcentaje mínimo de acierto que le exiges a la IA el umbral.

Subir foto e historial: Un apartado para meter imágenes desde tu ordenador si no quieres usar la cámara, y una lista abajo que va guardando lo que vas reconociendo.

2. El JavaScript app.js
Es el que hace todo el trabajo pesado tras bambalinas:

Cargar el modelo loadModel: Busca en la carpeta ./modelo/ los archivos de la IA model.json y metadata.json y los carga en la memoria.

Poner la cámara startCamera y loop: Pide permiso para usar la webcam, la enciende y crea un bucle un loop que no para de capturar fotogramas uno tras otro sin congelar la pantalla.

Adivinar qué hay predict: Le pasa la imagen a la IA. La IA le devuelve una lista con las probabilidades de cada clase que le enseñaste. El código busca la más alta, la compara con el porcentaje mínimo que pusiste en el cuadro de texto y, si pasa el filtro, escribe el resultado en pantalla.

La opción de subir archivo e historial: Si subes una foto, la lee con un FileReader, la muestra en chiquitito, hace exactamente la misma predicción que con la cámara y añade una línea nueva a la lista de abajo con la hora a la que lo hiciste.

3. El CSS styles.css
Es para que no parezca una página del año 2000 y se vea limpia estilo Google:

Variables :root: Guarda los colores típicos azul de botón, rojo de parar, gris de fondo para no tener que repetirlos todo el rato.

Ajuste para móviles: Hace que la caja no se salga de la pantalla en el móvil, que los botones se pongan uno debajo de otro si la pantalla es pequeña y que la imagen de la cámara no se deforme haciéndola cuadrada aspect-ratio: 1/1.

Efectos: Le da bordes redondeados a todo, sombras para que parezcan tarjetas flotantes y un pequeño efecto al pulsar los botones para que sientas que los estás apretando.

<!DOCTYPE html>
<html lang="es">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">

    <title>Clasificador de imágenes</title>

    <link rel="stylesheet" href="styles.css">

    <!-- TensorFlow.js -->
    <script src="https://cdn.jsdelivr.net/npm/@tensorflow/tfjs@latest"></script>

    <!-- Teachable Machine -->
    <script src="https://cdn.jsdelivr.net/npm/@teachablemachine/image@latest"></script>
</head>

<body>

    <div class="container">

        <h1>Clasificador de imágenes</h1>

        <p>
            Utiliza la cámara para clasificar una imagen mediante
            un modelo entrenado con Teachable Machine.
        </p>

        <!-- Cámara -->
        <div id="webcam-container"></div>

        <!-- Resultado -->
        <div class="resultado">
            <h2>Resultado</h2>

            <p id="resultado">
                Pulsa "Iniciar" para comenzar.
            </p>

            <p id="confianza">
                Confianza: -
            </p>
        </div>

        <!-- Botones -->
        <div class="botones">
            <button id="startButton">Iniciar</button>
            <button id="stopButton">Parar</button>
        </div>

        <!-- Umbral -->
        <div class="configuracion">

            <label for="threshold">
                Coincidencia mínima:
            </label>

            <input
                type="number"
                id="threshold"
                min="0"
                max="100"
                value="60"
            >

            <span>%</span>

        </div>

        <!-- Subir imagen -->
        <div class="subir-imagen">

            <h2>Subir una imagen</h2>

            <input
                type="file"
                id="imageUpload"
                accept="image/*"
            >

            <div>
                <img
                    id="preview"
                    alt="Vista previa de la imagen"
                >
            </div>

            <button id="classifyButton">
                Clasificar imagen
            </button>

        </div>

        <!-- Historial -->
        <div class="historial">

            <h2>Historial de resultados</h2>

            <ul id="historyList"></ul>

            <button id="clearHistory">
                Borrar historial
            </button>

        </div>

    </div>

    <script src="app.js"></script>

</body>
</html>

let model;
let webcam;
let running = false;

const MODEL_URL = "./modelo/model.json";
const METADATA_URL = "./modelo/metadata.json";

const startButton = document.getElementById("startButton");
const stopButton = document.getElementById("stopButton");

const resultText = document.getElementById("resultado");
const confidenceText = document.getElementById("confianza");

const thresholdInput = document.getElementById("threshold");

const imageUpload = document.getElementById("imageUpload");
const preview = document.getElementById("preview");
const classifyButton = document.getElementById("classifyButton");

const historyList = document.getElementById("historyList");
const clearHistoryButton = document.getElementById("clearHistory");



async function loadModel() {

    try {

        resultText.textContent = "Cargando modelo...";

        model = await tmImage.load(
            MODEL_URL,
            METADATA_URL
        );

        resultText.textContent = "Modelo cargado. Pulsa Iniciar.";

        console.log("Modelo cargado correctamente.");

    } catch (error) {

        console.error(error);

        resultText.textContent =
            "Error al cargar el modelo.";

    }

}




async function startCamera() {

    if (!model) {
        await loadModel();
    }

    if (running) {
        return;
    }

    try {

        webcam = new tmImage.Webcam(
            400,
            400,
            true
        );

        await webcam.setup();
        await webcam.play();

        running = true;

        document
            .getElementById("webcam-container")
            .innerHTML = "";

        document
            .getElementById("webcam-container")
            .appendChild(webcam.canvas);

        window.requestAnimationFrame(loop);

        resultText.textContent = "Clasificando...";

    } catch (error) {

        console.error(error);

        resultText.textContent =
            "No se pudo acceder a la cámara.";

    }

}



async function loop() {

    if (!running) {
        return;
    }

    webcam.update();

    await predict(webcam.canvas);

    window.requestAnimationFrame(loop);

}




async function predict(image) {

    if (!model) {
        return;
    }

    const predictions = await model.predict(image);

    let bestPrediction = predictions[0];

    for (let i = 1; i < predictions.length; i++) {

        if (
            predictions[i].probability >
            bestPrediction.probability
        ) {

            bestPrediction = predictions[i];

        }

    }

    const confidence =
        bestPrediction.probability * 100;

    const threshold =
        Number(thresholdInput.value);

    confidenceText.textContent =
        `Confianza: ${confidence.toFixed(2)}%`;

    if (confidence >= threshold) {

        resultText.textContent =
            `Resultado: ${bestPrediction.className}`;

    } else {

        resultText.textContent =
            "No hay coincidencia suficiente.";

    }

}




function stopCamera() {

    running = false;

    if (webcam) {

        webcam.stop();

        document
            .getElementById("webcam-container")
            .innerHTML = "";

    }

    resultText.textContent =
        "Clasificación detenida.";

    confidenceText.textContent =
        "Confianza: -";

}




imageUpload.addEventListener(
    "change",
    function(event) {

        const file = event.target.files[0];

        if (!file) {
            return;
        }

        const reader = new FileReader();

        reader.onload = function(e) {

            preview.src = e.target.result;

        };

        reader.readAsDataURL(file);

    }
);



classifyButton.addEventListener(
    "click",
    async function() {

        if (!model) {

            await loadModel();

        }

        if (!preview.src) {

            resultText.textContent =
                "Primero selecciona una imagen.";

            return;

        }

        const predictions =
            await model.predict(preview);

        let bestPrediction =
            predictions[0];

        for (
            let i = 1;
            i < predictions.length;
            i++
        ) {

            if (
                predictions[i].probability >
                bestPrediction.probability
            ) {

                bestPrediction =
                    predictions[i];

            }

        }

        const confidence =
            bestPrediction.probability * 100;

        const threshold =
            Number(thresholdInput.value);

        confidenceText.textContent =
            `Confianza: ${confidence.toFixed(2)}%`;

        if (confidence >= threshold) {

            resultText.textContent =
                `Resultado: ${bestPrediction.className}`;

            addToHistory(
                bestPrediction.className,
                confidence
            );

        } else {

            resultText.textContent =
                "No hay coincidencia suficiente.";

        }

    }
);




function addToHistory(result, confidence) {

    const date = new Date();

    const time =
        date.toLocaleTimeString();

    const item =
        document.createElement("li");

    item.textContent =
        `${time} - ${result} (${confidence.toFixed(2)}%)`;

    historyList.prepend(item);

}



clearHistoryButton.addEventListener(
    "click",
    function() {

        historyList.innerHTML = "";

    }
);




startButton.addEventListener(
    "click",
    startCamera
);

stopButton.addEventListener(
    "click",
    stopCamera
);



loadModel();

* {
  box-sizing: border-box;
  margin: 0;
  padding: 0;
}

:root {
  --primary-color: #1a73e8;
  --primary-hover: #1557b0;
  --danger-color: #d93025;
  --danger-hover: #b31412;
  --neutral-bg: #f8f9fa;
  --card-bg: #ffffff;
  --text-main: #202124;
  --text-muted: #5f6368;
  --border-color: #dadce0;
  --radius: 12px;
}


body {
  font-family: 'Google Sans', Roboto, Helvetica, Arial, sans-serif;
  background-color: var(--neutral-bg);
  color: var(--text-main);
  display: flex;
  justify-content: center;
  align-items: center;
  min-height: 100vh;
  padding: 16px;
  width: 100%;
}


.container {
  width: 100%;
  max-width: 720px;
  background: var(--card-bg);
  padding: 24px;
  border-radius: var(--radius);
  box-shadow: 0 4px 24px rgba(0, 0, 0, 0.08);
}

h1 {
  font-size: clamp(20px, 4vw, 26px);
  font-weight: 500;
  color: var(--primary-color);
  margin-bottom: 8px;
}

p {
  font-size: clamp(13px, 2vw, 14px);
  color: var(--text-muted);
  line-height: 1.5;
  margin-bottom: 16px;
}

h2 {
  font-size: clamp(16px, 3vw, 18px);
  font-weight: 500;
  color: #3c4043;
  margin-top: 16px;
  margin-bottom: 8px;
}

#webcam-container {
  display: flex;
  justify-content: center;
  align-items: center;
  width: 100%;
  margin: 12px 0;
  overflow: hidden;
}

#webcam-container canvas {
  max-width: 100% !important;
  width: 100% !important;
  height: auto !important;
  aspect-ratio: 1 / 1;
  object-fit: cover;
  border-radius: 8px;
  border: 1px solid var(--border-color);
}

#preview {
  display: none;
  max-width: 100%;
  width: 180px;
  height: 180px;
  object-fit: cover;
  border-radius: 8px;
  border: 1px solid var(--border-color);
  margin: 8px 0;
}


.resultado {
  background-color: #f1f3f4;
  padding: 14px 18px;
  border-radius: 8px;
  margin: 16px 0;
}

.resultado h2 {
  margin-top: 0;
  font-size: 14px;
  color: var(--text-muted);
  text-transform: uppercase;
  letter-spacing: 0.5px;
}

#resultado {
  font-size: clamp(18px, 3.5vw, 22px);
  font-weight: 600;
  color: var(--text-main);
  word-break: break-word;
}

#confianza {
  font-size: 13px;
  color: var(--text-muted);
  margin-top: 4px;
}


.botones {
  display: flex;
  flex-wrap: wrap;
  gap: 10px;
  margin: 16px 0;
}

button {
  background-color: var(--primary-color);
  color: #ffffff;
  border: none;
  border-radius: 6px;
  padding: 10px 20px;
  cursor: pointer;
  font-size: 14px;
  font-weight: 500;
  flex: 1;
  min-width: 120px;
  transition: background-color 0.2s, box-shadow 0.2s;
  touch-action: manipulation;
}

button:hover {
  background-color: var(--primary-hover);
  box-shadow: 0 2px 6px rgba(0, 0, 0, 0.12);
}

button:active {
  transform: scale(0.98);
}

#stopButton {
  background-color: var(--danger-color);
}

#stopButton:hover {
  background-color: var(--danger-hover);
}

#clearHistory {
  background-color: var(--text-muted);
  width: 100%;
  margin-top: 10px;
  flex: none;
}

#clearHistory:hover {
  background-color: #3c4043;
}


.configuracion {
  display: flex;
  align-items: center;
  flex-wrap: wrap;
  gap: 8px;
  margin: 16px 0;
  font-size: 14px;
  color: #3c4043;
}

input[type="number"] {
  width: 70px;
  padding: 8px;
  border: 1px solid var(--border-color);
  border-radius: 6px;
  font-size: 14px;
  text-align: center;
  background-color: #fff;
}

input[type="number"]:focus {
  border-color: var(--primary-color);
  outline: none;
}

.subir-imagen, .historial {
  border-top: 1px solid #e8eaed;
  padding-top: 16px;
  margin-top: 16px;
}

input[type="file"] {
  font-size: 13px;
  color: var(--text-muted);
  margin: 8px 0;
  width: 100%;
  padding: 6px;
  background: #f8f9fa;
  border-radius: 6px;
  border: 1px dashed var(--border-color);
}

ul {
  list-style-type: none;
  padding-left: 0;
  max-height: 140px;
  overflow-y: auto;
  margin-top: 8px;
}

li {
  background: #f8f9fa;
  padding: 8px 12px;
  border-radius: 6px;
  margin-bottom: 6px;
  font-size: 13px;
  border-left: 3px solid var(--primary-color);
  word-break: break-word;
}


@media screen and (max-width: 480px) {
  body {
    padding: 8px;
    align-items: flex-start;
  }

  .container {
    padding: 14px;
    border-radius: 8px;
    box-shadow: none;
  }

  .botones {
    flex-direction: column;
  }

  button {
    width: 100%;
    min-width: 100%;
    padding: 12px;
  }

  .configuracion {
    flex-direction: row;
    justify-content: space-between;
  }
}